# Suite EVL — Simplifiers

`Simplifiers.OfAny(term, scope)` simplifies a term with the variables in `scope` bound: what they decide folds into
literals, and the rest stays, the residual, a Patterns term. Its root is a literal when the scope decides the term:
`value` is then its value, and `evaluate()` gives it, or raises `NotEvaluated` naming why the root is not one. A term is
a value of `Expressions.Framework.Term`, built in a store, in a transaction (here through `term`, from a writer).
Simplification is three-valued (absent and incomparable values are unknown, and logic follows Kleene), never coerces
native types, and folds only the core operations: any other stays in the residual, named (`residues`).

In [ ]:
import { Framework } from "@mbse/expressions";
import { Simplifiers as V, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import { Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { KeyError, NotImplementedError, ValueError } = Errors;

const store: any = new Builders.Store();
store.transaction(Framework.install);
store.transaction(ISOIEC11404.install); // the standard dialects, before the dialects that use their kinds
store.transaction(IEEE754.install);
store.transaction(IEEE1164.install);
store.transaction(Unicode.install);
store.transaction(Patterns.install);
store.transaction(Patterns.define); // Patterns' operations, which validation reads

/** The term `spec` writes, the root of a holder built within the transaction open: validation reads its holder's
 * store, and the term, a visitor of that transaction, lives within it. */
function term(spec: unknown): any {
  return F.term(store.Expressions.Framework.Expression((e: any) => e.root(E.DIALECT.spec(spec))));
}

/** The value the simplification of the term `spec` writes folds to, with `scope` bound: null where its root is not a
 * literal, unknown. */
const ev = (spec: unknown, scope: Record<string, unknown> = {}): any => V.OfAny(term(spec), scope).value;

/** Whether a simplification's residual is the term `expected` writes. */
const same_ = (simplified: any, expected: unknown): boolean => F.same(term(simplified.residual), term(expected));

/** A person, a reference object; a card that reaches its holder by phone or email, a union of value objects. */
function people(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("evl.people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Reach: S.OfUnion, Code: S.OfUnion, Card: S.OfObject })
    .spec([
      (_: any) => _.Person.name("evl.Person").ref(true).properties({ name: S.OfStr, age: S.OfInt }),
      (_: any) => _.Reach.name("evl.Reach").branches([
        (t: any) => t.name("phone").schema([S.OfObject((o: any) => o.properties({ kind: S.OfStr, number: S.OfStr }))]),
        (t: any) => t.name("email").schema([S.OfObject((o: any) => o.properties({ kind: S.OfStr, address: S.OfStr }))]),
      ]),
      (_: any) => _.Code.name("evl.Code").branches([ // a union of scalars
        (t: any) => t.name("number").schema([S.OfInt]), (t: any) => t.name("text").schema([S.OfStr]),
        (t: any) => t.name("digits").schema([S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt]))])]),
      (_: any) => _.Card.name("evl.Card").ref(true).properties({ reach: _.Reach, code: _.Code }),
    ])]))]);
}

store.transaction(people);

/** `ann`, a person aged 30, and `nobody`, a person with no properties, the persons the cases below share, built anew
 * within the transaction open, as an object read or built lives only within the transaction that made it. */
function persons(): [any, any] {
  return [store.evl.Person((p: any) => p.name("Ann").age(30n)), store.evl.Person((p: any) => p)];
}

<a id="EVL-01"></a>
## EVL-01 · Simplifying literals, variables, lets, `get` and `has`

In [ ]:
let person: any;
store.transaction(() => {
  const [ann, nobody] = persons();
  person = E.variable("person");
  assert(ev(3n) === 3n && ev(E.literal("x")) === "x");
  const bound = V.OfAny(term(person), { person: ann }); // an object has no literal: the variable stays, its value known
  assert(!bound.literal && bound.value === null && same_(bound, person) && same([...bound.free], ["person"]));
  assert(ev(person.age, { person: ann }) === 30n && ev(person.age, { person: nobody }) === null); // absent: unknown
  assert(ev(person.has("age"), { person: ann }) === true && ev(person.has("age"), { person: nobody }) === false);
  assert(ev(person.age, { person: null }) === null && ev(person.has("age"), { person: null }) === null); // unknown object
  assert(ev(E.let_("a", person.age, E.variable("a").add(1n)), { person: ann }) === 31n);
  assert(ev(E.let_("a", 1n, E.let_("a", 2n, E.variable("a")))) === 2n); // the innermost binding wins
  assert(ev(E.let_("a", 1n, E.variable("a").add(E.let_("a", 2n, E.variable("a"))))) === 3n); // and only in its body
  const [one, term_value] = [term(E.literal(1n)), E.variable("term_value")];
  assert(ev(term_value.has("patterns"), { term_value: one }) === true); // terms are values too: a union value of their dialect
  assert(ev(term_value.patterns.literal.value, { term_value: one }) === 1n);
  const unbound = V.OfAny(term(E.variable("x"))); // not bound: it stays, and its root is no literal
  assert(unbound.value === null && same_(unbound, E.variable("x")) && unbound.residues.length === 0);
  raises(NotEvaluated, () => unbound.evaluate(), "the root is not a closed constructor expression: x is not bound");
  assert(V.OfAny(term(E.literal(1n).add(2n))).evaluate() === 3n); // a literal root: evaluated
  raises(TypeError, () => ev(E.operation("get", 1n, "x")), "get expects an object, got int");
  raises(TypeError, () => ev(E.operation("has", person, 1n), { person: ann }), "has expects a property name, got int");
});

<a id="EVL-02"></a>
## EVL-02 · Comparisons are three-valued and never coerce

In [ ]:
store.transaction(() => {
  const [ann, nobody] = persons();
  const unit = E.literal(1n);
  assert(same([ev(unit.lt(2n)), ev(unit.le(1n)), ev(unit.gt(2n)), ev(unit.ge(1n)), ev(unit.eq(1n)), ev(unit.ne(1n))], [
    true, true, false, true, true, false]));
  assert(ev(E.literal("b").gt("a")) === true && ev(E.literal(new Uint8Array([1])).lt(new Uint8Array([2]))) === true);
  assert(ev(unit.eq(1.0)) === null && ev(unit.lt(1.5)) === null && ev(unit.ne("1")) === null); // different types
  assert(ev(E.literal(true).eq(false)) === false && ev(E.literal(true).lt(false)) === null); // bools are not ordered
  assert(ev(E.literal(true).le(true)) === true);
  const nan = E.literal(NaN);
  assert(ev(nan.eq(NaN)) === true && ev(nan.eq(1.0)) === false && ev(nan.lt(1.0)) === null);
  assert(ev(E.literal(-0).lt(0)) === true && ev(E.literal(-0).eq(0)) === false);
  const age = E.variable("x").age;
  assert(ev(age.eq(1n), { x: nobody }) === null && ev(age.ne(1n), { x: nobody }) === null && ev(age.lt(1n), { x: nobody }) === null);
  const [first, second] = [E.variable("a"), E.variable("b")];
  assert(ev(first.eq(second), { a: ann, b: ann }) === true && ev(first.eq(second), { a: ann, b: nobody }) === false); // objects: by identity
  assert(ev(first.lt(second), { a: ann, b: nobody }) === null && ev(first.eq(1n), { a: ann }) === null);
});

<a id="EVL-03"></a>
## EVL-03 · Logic follows Kleene; arithmetic keeps one type

In [ ]:
store.transaction(() => {
  const [T, F_, U] = [E.literal(true), E.literal(false), E.variable("u")]; // u is unknown

  function table(method: string): unknown[][] {
    return [T, F_, U].map((x) => [T, F_, U].map((y) => ev((x as any)[method](y), { u: null })));
  }

  assert(equal(table("and_"), [[true, false, null], [false, false, false], [null, false, null]]));
  assert(equal(table("or_"), [[true, true, true], [true, false, null], [true, null, null]]));
  assert(equal(table("implies"), [[true, false, null], [true, true, true], [true, null, null]]));
  assert(same([T, F_, U].map((x) => ev(x.not_(), { u: null })), [false, true, null]));
  const missing = E.variable("missing"); // never evaluated when the first operand decides
  assert(ev(F_.and_(missing)) === false && ev(T.or_(missing)) === true && ev(F_.implies(missing)) === true);
  raises(TypeError, () => ev(T.and_(1n)), "and expects bool operands, got int");
  raises(TypeError, () => ev(E.literal("x").not_()), "not expects bool operands, got str");
  const pair = E.literal(2n);
  assert(same([ev(pair.add(3n)), ev(pair.sub(3n)), ev(pair.mul(3n)), ev(pair.neg())], [5n, -1n, 6n, -2n]));
  assert(ev(E.literal(1.5).add(1.0)) === 2.5 && ev(E.literal(2n ** 70n).mul(2n)) === 2n ** 71n);
  assert(ev(U.add(1n), { u: null }) === null && ev(U.neg(), { u: null }) === null);
  raises(TypeError, () => ev(E.literal(1n).add(1.5)), "add expects numbers of one domain, got int and float");
  raises(TypeError, () => ev(T.sub(T)), "sub expects numbers of one domain, got bool and bool");
  raises(TypeError, () => ev(E.literal("x").neg()), "neg expects a number, got str");
});

<a id="EVL-04"></a>
## EVL-04 · Checking: `validate()` finds what simplification would leave open or raise

In [ ]:
store.transaction(() => {
  assert(same(E.DIALECT.validate(term(E.operation("lt", 1n))), ["lt takes 2 arguments, got 1"]));
  const foo = term(E.operation("foo", 1n));
  assert(same(E.DIALECT.validate(foo), ["'foo' is not an operation Patterns defines"]) && same(E.DIALECT.validate(foo, [], true), ["'foo' is not an operation Patterns defines"]));
  person = E.variable("person");
  const checked = term(E.let_("a", person.age, E.variable("a").ge(18n).and_(person.has("email"))));
  assert(E.DIALECT.validate(checked, ["person"], true).length === 0);
  assert(same(E.DIALECT.validate(checked), ["value: argument 0: variable 'person' is not bound",
    "body: argument 1: argument 0: variable 'person' is not bound"]));
  const kept = V.OfAny(foo); // an operation Patterns does not define stays, named
  assert(same_(kept, E.operation("foo", 1n)) && same(kept.residues.map((r) => r.reason), ["Patterns call 'foo' is not simplified"]));
  raises(NotEvaluated, () => kept.evaluate(), "the root is not a closed constructor expression: Patterns call 'foo' is not simplified");
  raises(TypeError, () => ev(E.operation("lt", 1n)), "lt takes 2 arguments, got 1");
  raises(ValueError, () => ev(E.literal(null)), "a literal needs a value");
  raises(ValueError, () => ev(E.let_("x", null, null)), "a let needs a value");
  raises(ValueError, () => ev(E.let_("x", 1n, null)), "a let needs a body");
  // A term is a nested value: written into another, it is copied, so a term never holds itself.
  const inner = term(E.operation("not", true));
  const outer = term(E.operation("not", inner));
  assert(ev(outer) === true && [...F.walk(outer)].length === 3 && [...F.walk(inner)].length === 2);
  raises(TypeError, () => E.operation("not", [7n]), "expected a term, a writer or a native value, got [7]");
});

<a id="EVL-05"></a>
## EVL-05 · The union of terms names each branch by its dialect, and a dialect's `Term` each by its kind's tag

In [ ]:
store.transaction(() => {
  const [Term, Union]: any[] = [store.registered("Expressions.Framework.Term"), store.registered("Expressions.Patterns.Expressions.Term")];
  assert(same(Term.branches.map((branch: any) => branch.name), ["patterns"]) && Term.branches[0].schema === Union);
  assert(same(Union.branches.map((branch: any) => branch.name), [
    "literal", "call", "variable", "let", "list", "mapping", "quantifier", "error", "missing"]));
  assert(same(Union.branches.map((branch: any) => branch.schema.name), [
    "Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
    "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping",
    "Expressions.Patterns.Expressions.Quantifier", "Expressions.Framework.Error", "Expressions.Framework.Missing"]));
  const samples = [E.literal(1n), E.operation("f", 1n, "x"), E.variable("x"), E.let_("a", 1n, 2n), E.list_(1n), E.mapping(["k", 1n]),
    E.quantifier("all", "x", E.variable("xs"), true), (u: any) => u.error((r: any) => r.text("?")),
    (u: any) => u.missing((r: any) => r)];
  assert(same(samples.map((sample) => F.kindOf(term(sample))[0]), Union.branches.map((branch: any) => branch.name)));
  assert(Union.validate().length === 0);
  const invalid = [V.OfAny(term(samples[7])), V.OfAny(term(samples[8]))]; // invalid syntax stays, named: no value of its own
  assert(equal(invalid.flatMap((s) => s.residues.map((r) => [r.dialect, r.kind, r.category])), [
    ["Patterns", "error", "b"], ["Patterns", "missing", "b"]]));
  assert(same(invalid.map((s) => (s.residues[0] as any).reason), ["Patterns error is not simplified", "Patterns missing is not simplified"]));
});

<a id="EVL-06"></a>
## EVL-06 · One entry point simplifies a term of any kind, an expression or its root

In [ ]:
store.transaction(() => {
  const x = E.variable("x");
  assert(ev(3n) === 3n && ev(E.literal("s")) === "s" && ev(x, { x: 2n }) === 2n);
  assert(ev(x.add(1n), { x: 1n }) === 2n && ev(E.let_("y", 1n, E.variable("y").neg())) === -1n);
  assert(ev(x.all("i", E.variable("i").gt(0n)), { x: [1n, 2n] }) === true);
  assert(ev(x, { x: null }) === null);
  const expression = store.Expressions.Framework.Expression((e: any) => e.root(x.add(1n)));
  assert(V.OfAny(expression, { x: 1n }).value === 2n && V.OfAny(expression.root, { x: 2n }).value === 3n); // an expression, its root
  raises(TypeError, () => V.OfAny(3n), "not a term: 3");
  // A value that a dialect writes as a closed constructor expression, not a literal, is an evaluation too
  const built = new Simplification.Simplifier(V.RULES, { literal: () => null, construct: (value: unknown) => E.literal(value) });
  const known = built.known(x, 3n);
  assert(known.constructed && !known.literal && known.value === 3n);
  const simplified = built.run(term(x.add(1n)), { x: 1n });
  assert(simplified.constructed && !simplified.literal && simplified.evaluate() === 2n);
});

<a id="EVL-07"></a>
## EVL-07 · A constraint about a value, bound to a variable, reads value objects and union values

In [ ]:
store.transaction(() => {
  const [card, mail] = [E.variable("card"), E.variable("mail")]; // a card, and an email, a value object
  const [good, phone, sameCard, home, workPhone] = [
    store.evl.Card((c: any) => c.reach((r: any) => r.email((e: any) => e.kind("work").address("a@x")))),
    store.evl.Card((c: any) => c.reach((r: any) => r.phone((e: any) => e.number("1")))),
    store.evl.Card((c: any) => c.reach((r: any) => r.email((e: any) => e.kind("work").address("a@x")))),
    store.evl.Card((c: any) => c.reach((r: any) => r.email((e: any) => e.kind("home").address("a@x")))),
    store.evl.Card((c: any) => c.reach((r: any) => r.phone((e: any) => e.kind("work")))),
  ];
  const email = card.get("reach").get("email"); // a union value is a record of its one branch
  assert(ev(card.get("reach").has("email"), { card: good }) === true && ev(card.get("reach").has("phone"), { card: good }) === false);
  assert(ev(email.get("kind").eq("work"), { card: good }) === true);
  assert(ev(email.get("kind").eq("home"), { card: phone }) === null);
  assert(ev(mail.get("kind").eq("phone"), { mail: good.reach.email }) === false && ev(mail.has("kind"), { mail: null }) === null);
  assert(ev(mail.get("kind"), { mail: good.reach.email }) === "work"); // a value object's property
  assert(ev(mail.eq(mail), { mail: good.reach.email }) === true); // a value object compares deeply, by its properties
  const other = E.variable("other");
  const reach = card.get("reach").eq(other.get("reach")); // union values: the same branch, equal
  assert(ev(reach, { card: good, other: sameCard }) === true);
  assert(ev(reach, { card: good, other: home }) === false);
  assert(ev(reach, { card: good, other: workPhone }) === null); // another branch
  assert(ev(card.eq(other), { card: good, other: good.reach.email }) === null && ev(card.eq(other), { card: good, other: good }) === true);
  const coded = store.evl.Card((c: any) => c.code((r: any) => r.number(7n)));
  assert(ev(card.get("code").get("number"), { card: coded }) === 7n && ev(card.get("code").has("text"), { card: coded }) === false);
  const listed = store.evl.Card((c: any) => c.code((r: any) => r.digits((k: any) => {
    k.at(0n).as_scalar(1n); // a container branch is written through its visitor
    k.at(1n).as_scalar(2n);
    return k;
  })));
  assert(ev(card.get("code").get("digits").count(), { card: listed }) === 2n); // a union's container branch reads as a collection
});

<a id="EVL-08"></a>
## EVL-08 · A parameter reference is a leaf: a context (`Contexts.OfContext`) assigns its parameter a value that folds, apart from the variables, or a term read in its place; one neither assigned nor bound stays, named among the free parameters; one bound to an object is put as its reference where a literal's value stood, and stays, named, in a term's place; `Terms.same` compares two by name

In [ ]:
import { JSON, References } from "@mbse/schemas/Framework";
import { Symbolics as S } from "@mbse/expressions/Framework";

/** A context (`Contexts.OfContext`) whose parameters are given by name: each assigned a value, or, given as
 * `[reference]`, bound to the object the reference designates. */
function context(given: Record<string, unknown>): any {
  const entry = (name: string, target: unknown) => (Array.isArray(target)
    ? (p: any) => p.name(name).object(target[0])
    : (p: any) => p.name(name).value((v: any) => v.value(target)));
  return store.Contexts.OfContext((c: any) => c.parameters(Object.fromEntries(
    Object.entries(given).map(([name, target]) => [name, entry(name, target)]))));
}

store.transaction(() => {
  const x = E.variable("x");
  const [n, m] = ["n", "m"].map((k) => store.References.OfParameter((p: any) => p.name(k)));
  // A parameter its context assigns a value is replaced by the value's literal, and what it closes folds; a context
  // is taken where a scope is, its variables the scope's variables
  assert(V.OfAny(term(x.add(n)), new S.Variables({ x: 1n }).within(context({ n: 2n }))).value === 3n);
  assert(same_(V.OfAny(term(x.add(n)), context({ n: 2n })), x.add(2n)) && V.OfAny(term(n), context({ n: 2n })).value === 2n);
  const both = store.Contexts.OfContext((c: any) => c.variables({ x: (v: any) => v.name("x").value((r: any) => r.value(5n)) })
    .parameters({ n: (p: any) => p.name("n").value((r: any) => r.value(2n)) }));
  assert(V.OfAny(term(x.add(n)), both).value === 7n);
  // A parameter and a variable of one name never collide: the variable n is 10, the parameter n is 1
  assert(V.OfAny(term(E.variable("n").add(n)), new S.Variables({ n: 10n }).within(context({ n: 1n }))).value === 11n);
  assert(V.OfAny(term(E.let_("n", 5n, E.variable("n").add(n))), context({ n: 1n })).value === 6n); // a let binds the variable
  // A parameter its context neither assigns nor binds stays, named among the free parameters, beside the free names
  const stays = V.OfAny(term(x.add(n).add(m)), new S.Variables({ x: 1n }).within(context({ m: 4n })));
  assert(same_(stays, E.literal(1n).add(n).add(4n)) && same([...stays.parameters], ["n"]) && stays.free.size === 0);
  assert(same([...V.OfAny(term(n)).parameters], ["n"]) && same_(V.OfAny(term(n)), n)); // at the root, written back there
  raises(NotEvaluated, () => stays.evaluate(),
    "the root is not a closed constructor expression: parameter 'n' is neither assigned nor bound");
  // A parameter assigned a term is read in its place, where its place holds that term (any term, in Patterns' calls)
  const plus = term(x.add(1n));
  assert(same_(V.OfAny(term(E.operation("mul", n, 2n)), context({ n: plus })), E.operation("mul", x.add(1n), 2n)));
  assert(V.OfAny(term(E.operation("mul", n, 2n)), new S.Variables({ x: 3n }).within(context({ n: plus }))).value === 8n);
  // A parameter bound to an object is substituted by the object's reference, put as it is where the parameter stood
  // in a literal's value, which holds any value: nothing resolved, nothing read
  const schema = References.of(store, store.registered("evl.Person")); // a schema object's reference: any object's would do
  const held = (u: any) => u.literal((r: any) => r.value(n));
  assert(E.DIALECT.validate(term(held)).length === 0 && [...V.OfAny(term(held)).parameters].join() === "n");
  assert(V.OfAny(term(held), context({ n: 2n })).value === 2n); // assigned a value: the literal the dialect writes of it
  const put = V.OfAny(term(E.list_(held, x)), context({ n: [schema] }));
  const expected = E.list_((u: any) => u.literal((r: any) => r.value(schema)), x);
  const json = (spec: unknown) => JSON.ToJSON(store)(store.registered("Expressions.Framework.Term"), term(spec));
  assert(json(put.residual) === json(expected) && json(put.residual).includes('"References.OfObject"'));
  assert(put.parameters.size === 0 && [...put.free].join() === "x" && put.residues.length === 0);
  assert(V.OfAny(term(E.literal(1n).add(held)), context({ n: 2n })).value === 3n); // a value assigned there folds
  // A term's place holds no object's reference: there the parameter stays, named by a residue of class (b)
  const bound = V.OfAny(term(x.add(n)), new S.Variables({ x: 1n }).within(context({ n: [schema] })));
  assert(same_(bound, E.literal(1n).add(n)) && bound.parameters.size === 0);
  assert(equal(bound.residues.map((r) => [r.reason, r.category]), [
    ["Patterns parameter 'n' bound to an object, which its place does not hold is not simplified", "b"]]));
  // A scope's sub-context: the inner context's parameters replace the outer one's of their names
  const inner = new S.Variables().within(context({ n: 2n, m: 3n })).within(context({ n: 4n }));
  assert(equal(S.given(inner, "n"), ["value", 4n]) && equal(S.given(inner, "m"), ["value", 3n]) && S.given(inner, "k") === null);
  assert(S.given({}, "n") === null && (new S.Variables().within(context({ n: [schema] })).parameters.get("n") as S.Given)[0] === "object");
  raises(TypeError, () => new S.Variables().within({ n: 2n }), "a scope is within a context (Contexts.OfContext), not ");
  // Terms.same compares parameter references by name: two references to n are the same, n and m differ
  assert(F.same(term(x.add(n)), term(x.add(store.References.OfParameter((p: any) => p.name("n"))))));
  assert(!F.same(term(x.add(n)), term(x.add(m))) && !F.same(term(x.add(n)), term(x.add(E.variable("n")))));
  // Read as a leaf of every dialect: a reference its scope resolves, which validation does not require bound
  const leaf = F.term(term(x.add(n)));
  const read = F.form(leaf).arguments[1];
  assert(F.isParameter(read) && F.form(read).kind === "parameter" && equal(F.form(read).attributes, new Map([["name", "n"]])));
  assert(F.form(read).arguments.length === 0);
  assert(E.DIALECT.validate(leaf, ["x"]).length === 0 && E.DIALECT.kind(read)[2] === F.PARAMETER_KIND);
  assert(F.dialectOf(term(n)) === "Expressions.Framework.Term" && E.DIALECT.kind(term(n))[0] === "parameter");
  raises(TypeError, () => term(new E.Writer((u: any) => F.spec(n)(u))),
    "a parameter reference is written in a term's place, not as a term's value");
  // A value with no literal, a union value, is known: what it closes folds, and the parameter itself stays, needing
  // nothing
  const code = store.evl.Card((c: any) => c.code((r: any) => r.number(4n))).code;
  const coded = V.OfAny(term(n), context({ n: code }));
  assert(same_(coded, n) && coded.parameters.size === 0 && coded.residues.length === 0);
  assert(V.OfAny(term(E.operation("get", n, "number")), context({ n: code })).value === 4n);
  // A parameter its context names but neither assigns nor binds is free; a parameter reference has no node
  const named = store.Contexts.OfContext((c: any) => c.parameters({ n: (p: any) => p.name("n") }));
  assert(S.given(new S.Variables().within(named), "n") === null && [...V.OfAny(term(n), named).parameters].join() === "n");
  raises(S.Unbound, () => F.node(...(F.kindOf(read) as [string, any])), "n");
});